In [2]:
import pandas as pd

,LOAD
1,7171000.0
2,6957000.0
3,6831000.0
4,6806000.0
5,6851000.0
...,...
8756,10727000.0
8757,10464000.0
8758,10152000.0
8759,9779000.0


In [41]:
load_bch_ = pd.to_numeric(load_bch[load_bch.columns[-1]], errors='coerce')

In [10]:
def fix_hourly_load(load, year):
    #Filter down to just the loads
    fixed = pd.to_numeric(load[load.columns[-1]], errors='coerce') #Rightmost column in BC Hydro hourly load spreadsheet, to_numeric converts the labels into NaN
    fixed = fixed.reset_index().drop(columns=['index'])
    fixed.columns = ['LOAD']

    #Remove NaN values (the column labels and sometimes the value for DST hour) and 0 values (sometimes the DST hour), then convert from MWh to kWh
    fixed = fixed.loc[fixed['LOAD'] > 0] * 1000
    
    #Index hourly loads by hourly timestamp
    fixed['TIME'] = pd.date_range(start=str(year)+'-01-01 00:00:00', end=str(year)+'-12-31 23:00:00', freq='h')
    fixed = fixed.set_index('TIME')

    #Return the hourly load for the entire province for a given year
    return fixed

In [11]:
data=fix_hourly_load(load_bch,2021)

In [15]:
data['LOAD_profile_norm'] = (data['LOAD'] - data['LOAD'].min()) / (data['LOAD'].max() - data['LOAD'].min())

In [33]:
import plotly.express as px
from pathlib import Path

title = f'Normalized Load Profile for {data.index[0].year}'
peak_load_time = data['LOAD'].idxmax()
peak_load_value = data['LOAD_profile_norm'].max()

fig = px.line(data, x=data.index, y='LOAD_profile_norm', title=title)
fig.add_scatter(x=[peak_load_time], y=[peak_load_value], mode='markers', marker=dict(color='red', size=10), name='Peak Load')
fig.update_layout(yaxis_title="Normalized Load Profile")
fig.show()
fig.write_html(str(Path('../vis') / title) + '.html')

In [39]:
import plotly.express as px
import pandas as pd
from pathlib import Path


title = f'Normalized Load Profile for {data.index[0].year}'
peak_load_time = data['LOAD'].idxmax()
peak_load_value = data['LOAD_profile_norm'].max()

# Function to resample data
def resample_data(freq):
    return data.resample(freq).mean()

# Create the initial figure
fig = px.line(data, x=data.index, y='LOAD_profile_norm', title=title)
fig.add_scatter(
    x=[peak_load_time],
    y=[peak_load_value],
    mode='lines',
    marker=dict(color='red', size=8),
    name='Peak Load'
)

# Customize layout for minimal and clean look
fig.update_layout(
    template='plotly_white',
    title={
        'text': title,
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18}
    },
    xaxis_title=None,
    yaxis_title="Normalized Load Profile",
    xaxis=dict(showgrid=False, ticks="outside"),
    yaxis=dict(showgrid=True, gridcolor="lightgrey"),
    font=dict(size=14),
    margin=dict(l=40, r=40, t=50, b=40),
    showlegend=False
)

# Add dropdown for resampling
freq_options = ['h','D', 'W', 'ME']  # Daily, Weekly, Monthly
updatemenus = [
    {
        "buttons": [
            {
                "label": f"Sampling: {freq}",
                "method": "update",
                "args": [
                    {"x": [resample_data(freq).index], "y": [resample_data(freq)['LOAD_profile_norm']]},
                    {"title": f"Normalized Load Profile (Resampled: {freq})"}
                ],
            }
            for freq in freq_options
        ],
        "direction": "down",
        "x": 0.8,
        "xanchor": "center",
        "y": 1.15,
        "yanchor": "top",
        "showactive": True,
        "pad": {"r": 10, "t": 10},
    }
]

fig.update_layout(updatemenus=updatemenus)

# Show and save the figure
fig.show()
fig.write_html(str(Path('../vis') / title) + '.html')